# GroupBy and Aggregation

The groupby operation is one of the most powerful features in Pandas. It implements the "split-apply-combine" pattern: split data into groups, apply a function to each group, and combine results. This is how you answer questions like "What's the average salary by department?"

## Learning Objectives

- Understand the split-apply-combine pattern
- Use `.groupby()` with single and multiple columns
- Apply aggregations with `.agg()` including multiple functions
- Use `.transform()` to broadcast results back to original shape
- Use `.filter()` to filter groups based on group properties

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

In [ ]:
# Load data
df = pd.read_csv('../../data/titanic.csv')
print(f"Shape: {df.shape}")
df.head()

## Split-Apply-Combine Pattern

The groupby operation follows three steps:
1. **Split**: Divide data into groups based on some criteria
2. **Apply**: Apply a function to each group independently
3. **Combine**: Combine results into a new data structure

In [ ]:
# Visualize split-apply-combine
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Sample data for visualization
sample_df = pd.DataFrame({
    'Group': ['A', 'A', 'B', 'B', 'C', 'C'],
    'Value': [10, 20, 30, 40, 50, 60]
})

# Step 1: Split
ax = axes[0]
colors = {'A': '#e74c3c', 'B': '#2ecc71', 'C': '#3498db'}
for i, row in sample_df.iterrows():
    ax.barh(i, 1, color=colors[row['Group']], edgecolor='black')
    ax.text(0.5, i, f"{row['Group']}: {row['Value']}", ha='center', va='center', fontsize=12)
ax.set_xlim(0, 1)
ax.set_title('1. SPLIT\nDivide into groups', fontsize=12)
ax.axis('off')

# Step 2: Apply
ax = axes[1]
y_pos = [0, 1, 2]
group_means = sample_df.groupby('Group')['Value'].mean()
for i, (group, mean) in enumerate(group_means.items()):
    ax.barh(i, 1, color=colors[group], edgecolor='black')
    ax.text(0.5, i, f"{group}: mean={mean:.0f}", ha='center', va='center', fontsize=12)
ax.set_title('2. APPLY\nCompute mean per group', fontsize=12)
ax.axis('off')

# Step 3: Combine
ax = axes[2]
result = group_means.reset_index()
for i, row in result.iterrows():
    ax.barh(i, row['Value']/60, color=colors[row['Group']], edgecolor='black')
    ax.text(row['Value']/120, i, f"{row['Group']}: {row['Value']:.0f}", ha='center', va='center', fontsize=12)
ax.set_title('3. COMBINE\nResult DataFrame', fontsize=12)
ax.axis('off')

plt.tight_layout()
plt.show()

## Basic GroupBy

In [ ]:
# GroupBy object (lazy - doesn't compute until you apply a function)
grouped = df.groupby('Pclass')
print(f"Type: {type(grouped)}")
print(f"Number of groups: {grouped.ngroups}")
print(f"Groups: {list(grouped.groups.keys())}")

In [ ]:
# Apply aggregation
print("Mean fare by class:")
print(df.groupby('Pclass')['Fare'].mean())

In [ ]:
# Multiple aggregations at once
print("\nMultiple aggregations:")
print(df.groupby('Pclass')['Fare'].agg(['mean', 'median', 'std', 'count']))

In [ ]:
# Group by multiple columns
print("\nSurvival rate by Class and Sex:")
print(df.groupby(['Pclass', 'Sex'])['Survived'].mean().round(3))

In [ ]:
# Unstack for better readability
survival_by_class_sex = df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack()
print("\nSurvival rate (unstacked):")
survival_by_class_sex

## The .agg() Method

In [ ]:
# Different aggregations for different columns
result = df.groupby('Pclass').agg({
    'Survived': 'mean',
    'Age': ['mean', 'median'],
    'Fare': ['mean', 'max'],
    'PassengerId': 'count'  # Count of passengers
})
print("Different aggregations per column:")
result

In [ ]:
# Named aggregations (Pandas 0.25+) - cleaner column names
result = df.groupby('Pclass').agg(
    survival_rate=('Survived', 'mean'),
    avg_age=('Age', 'mean'),
    avg_fare=('Fare', 'mean'),
    max_fare=('Fare', 'max'),
    passenger_count=('PassengerId', 'count')
).round(2)
print("Named aggregations:")
result

In [ ]:
# Custom aggregation functions
def range_func(x):
    return x.max() - x.min()

result = df.groupby('Pclass')['Age'].agg(['mean', 'std', range_func])
print("With custom function:")
result

## The .transform() Method

While `.agg()` reduces groups to single values, `.transform()` returns data with the same shape as the input. This is useful for broadcasting group statistics back to each row.

In [ ]:
# Calculate group means and broadcast back
df_copy = df.copy()
df_copy['Fare_class_mean'] = df.groupby('Pclass')['Fare'].transform('mean')

print("Original Fare vs Class Mean:")
df_copy[['Pclass', 'Fare', 'Fare_class_mean']].head(10)

In [ ]:
# Use transform for within-group normalization
df_copy['Fare_normalized'] = df.groupby('Pclass')['Fare'].transform(
    lambda x: (x - x.mean()) / x.std()
)

print("Normalized fare within each class:")
df_copy[['Pclass', 'Fare', 'Fare_normalized']].head(10)

In [ ]:
# Fill missing values with group mean
df_copy = df.copy()
df_copy['Age_filled'] = df.groupby('Pclass')['Age'].transform(
    lambda x: x.fillna(x.mean())
)

print(f"Original Age nulls: {df['Age'].isna().sum()}")
print(f"After fill with class mean: {df_copy['Age_filled'].isna().sum()}")

## The .filter() Method

Filter entire groups based on a group property.

In [ ]:
# Keep only classes with more than 200 passengers
filtered = df.groupby('Pclass').filter(lambda x: len(x) > 200)
print(f"Original rows: {len(df)}")
print(f"After filter (groups > 200): {len(filtered)}")
print(f"Classes kept: {filtered['Pclass'].unique()}")

In [ ]:
# Keep embarkation ports with survival rate > 0.4
filtered = df.groupby('Embarked').filter(lambda x: x['Survived'].mean() > 0.4)
print(f"Ports with survival rate > 0.4: {filtered['Embarked'].unique()}")

## Method Chaining with GroupBy

In [ ]:
# Chain operations for readable code
result = (
    df
    .groupby(['Pclass', 'Sex'])
    .agg(
        survival_rate=('Survived', 'mean'),
        avg_age=('Age', 'mean'),
        count=('PassengerId', 'count')
    )
    .round(2)
    .sort_values('survival_rate', ascending=False)
)
print("Chained groupby analysis:")
result

## Visualization: Grouped Bar Chart

In [ ]:
# Survival rate by Class and Sex
survival = df.groupby(['Pclass', 'Sex'])['Survived'].mean().unstack()

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(survival.index))
width = 0.35

bars1 = ax.bar(x - width/2, survival['female'], width, label='Female', color='#e74c3c')
bars2 = ax.bar(x + width/2, survival['male'], width, label='Male', color='#3498db')

ax.set_xlabel('Passenger Class')
ax.set_ylabel('Survival Rate')
ax.set_title('Titanic Survival Rate by Class and Sex')
ax.set_xticks(x)
ax.set_xticklabels(['1st Class', '2nd Class', '3rd Class'])
ax.legend()
ax.set_ylim(0, 1)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.1%}',
            ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height, f'{height:.1%}',
            ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Exercises

### Exercise 1: Basic Aggregation

Calculate for each embarkation port (Embarked):
1. The number of passengers
2. The average fare
3. The survival rate

In [ ]:
# Calculate statistics by embarkation port
# YOUR CODE HERE

### Exercise 2: Transform Practice

Create a new column 'Fare_pct_of_class' that shows what percentage of the class's total fare each passenger paid.

In [ ]:
# Calculate fare percentage within class
# YOUR CODE HERE

### Exercise 3: Complex Grouping

Find the class (Pclass) that has:
1. The highest average age
2. The most variation in fare (highest standard deviation)
3. The best survival rate among males

In [ ]:
# Answer the questions using groupby
# YOUR CODE HERE

### Exercise 4: Visualization

Create a grouped bar chart showing the average age by Pclass and Survived status.

In [ ]:
# Create the visualization
# YOUR CODE HERE